In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames and keep them in kernel variables
df_metric = pd.read_csv("metric.csv", low_memory=False)
df_trace = pd.read_csv("trace.csv", low_memory=False)
df_log = pd.read_csv("log.csv", low_memory=False)
df_error = pd.read_csv("error_logs.csv", low_memory=False)

# Ensure timestamp -> timezone-aware datetimes in UTC+8 (Asia/Shanghai) as required
tz = pytz.timezone('Asia/Shanghai')

# Helper to add a tz-aware datetime column (named 'ts') without modifying original numeric timestamp
def add_tz_ts(df):
    if 'timestamp' in df.columns:
        # convert seconds to datetime and localize to Asia/Shanghai (treat epoch seconds as local time in UTC+8)
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
    else:
        df['ts'] = pd.NaT
    return df

df_metric = add_tz_ts(df_metric)
df_trace = add_tz_ts(df_trace)
df_log = add_tz_ts(df_log)
df_error = add_tz_ts(df_error)

# Helper to create compact summary for a generic dataframe
def compact_summary(df, name_col='cmdb_id', extra_name_col=None, extra_label=None, message_mode=False):
    rows = int(df.shape[0])
    if rows > 0:
        min_ts = df['ts'].min()
        max_ts = df['ts'].max()
    else:
        min_ts = None
        max_ts = None
    # top cmdb_id values
    if name_col in df.columns:
        top_cmdb = df[name_col].value_counts().head(20).rename_axis(name_col).reset_index(name='count')
    else:
        top_cmdb = pd.DataFrame(columns=[name_col, 'count'])
    result = {
        'rows': rows,
        'min_ts': min_ts,
        'max_ts': max_ts,
        'top_20_cmdb_id': top_cmdb
    }
    # extra fields for specific files
    if extra_name_col is not None and extra_name_col in df.columns:
        vc = df[extra_name_col].value_counts()
        result[f'top_20_{extra_name_col}'] = vc.head(20).rename_axis(extra_name_col).reset_index(name='count')
        result[f'distinct_{extra_name_col}'] = int(vc.shape[0])
    if message_mode and 'message' in df.columns:
        result['top_20_messages'] = df['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')
    return result

# Build summaries
summary_metric = compact_summary(df_metric, name_col='cmdb_id', extra_name_col='kpi_name')
summary_trace = compact_summary(df_trace, name_col='cmdb_id', extra_name_col='trace_name')
summary_log = compact_summary(df_log, name_col='cmdb_id', extra_name_col='log_name')
summary_error = compact_summary(df_error, name_col='cmdb_id', message_mode=True)

# Display compact summaries (variables shown as output)
summary_metric, summary_trace, summary_log, summary_error

```
Out[1]:
```
```python
# Compose a concise plain-English summary using the previously computed summary_... variables
# (These variables were created in the prior step and remain available in the kernel.)

def fmt_ts(ts):
    return ts.strftime('%Y-03-%d %H:%M:%S %Z') if ts is not None else 'N/A'

# Extract key pieces
m = summary_metric
t = summary_trace
l = summary_log
e = summary_error

top_m_cmdb = ", ".join(m['top_20_cmdb_id'].head(5)['cmdb_id'].astype(str).tolist())
top_m_kpis_count = m['distinct_kpi_name']

top_t_cmdb = ", ".join(t['top_20_cmdb_id'].head(5)['cmdb_id'].astype(str).tolist())
top_t_traces_count = t['distinct_trace_name']

top_l_cmdb = ", ".join(l['top_20_cmdb_id'].head(5)['cmdb_id'].astype(str).tolist())
top_l_lognames_count = l['distinct_log_name']

top_e_cmdb = ", ".join(e['top_20_cmdb_id'].head(5)['cmdb_id'].astype(str).tolist())
top_e_messages_sample = e['top_20_messages'].head(1)['message'].iloc[0] if not e['top_20_messages'].empty else 'N/A'

summary = (
    "Compact telemetry summary (timestamps shown in UTC+8):\n\n"
    f"1) metric.csv: {m['rows']} rows; time range {fmt_ts(m['min_ts'])} to {fmt_ts(m['max_ts'])}.\n"
    f"   - Top cmdb_id (top 5): {top_m_cmdb}.\n"
    f"   - Distinct kpi_name: {top_m_kpis_count}.\n\n"
    f"2) trace.csv: {t['rows']} rows; time range {fmt_ts(t['min_ts'])} to {fmt_ts(t['max_ts'])}.\n"
    f"   - Top cmdb_id (top 5): {top_t_cmdb}.\n"
    f"   - Distinct trace_name: {top_t_traces_count}.\n\n"
    f"3) log.csv: {l['rows']} rows; time range {fmt_ts(l['min_ts'])} to {fmt_ts(l['max_ts'])}.\n"
    f"   - Top cmdb_id (top 5): {top_l_cmdb}.\n"
    f"   - Distinct log_name count: {top_l_lognames_count} (main log names: see top counts).\n\n"
    f"4) error_logs.csv: {e['rows']} rows; time range {fmt_ts(e['min_ts'])} to {fmt_ts(e['max_ts'])}.\n"
    f"   - Top cmdb_id (up to 5): {top_e_cmdb}.\n"
    f"   - Most frequent message (sample): \"{top_e_messages_sample}\" (all 975 error rows share similar warning messages).\n\n"
    "Notes: DataFrames remain available in kernel variables df_metric, df_trace, df_log, df_error, and\n"
    "the compact summary objects summary_metric, summary_trace, summary_log, summary_error."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'rows': 147972, 'min_ts': Timestamp('2022-03-21 01:30:00+0800', tz='Asia/Shanghai'), 'max_ts': Timestamp('2022-03-21 01:59:00+0800', tz='Asia/Shanghai'), 'top_20_cmdb_id':                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5280
3                 frontend-1   5040
4                 frontend-0   4920
5                 frontend-2   4620
6          checkoutservice-0   3900
7          checkoutservice-2   3660
8          checkoutservice-1   3540
9         checkoutservice2-0   3540
10   productcatalogservice-1   3480
11   productcatalogservice-2   3480
12   productcatalogservice-0   3480
13              adservice2-0   3060
14             cartservice-1   3000
15  productcatalogservice2-0   2880
16             cartservice-0   2880
17            cartservice2-0   2880
18             cartservice-2   2640
19               adservice-0   2580, 'top_20_kpi_name':                                              kpi_name  count
0   container.node-5.container_network_transmit_pa...   1050
1   container.node-5.container_network_transmit_pa...   1050
2   container.node-5.container_network_transmit_er...   1050
3   container.node-5.container_network_transmit_MB...   1050
4   container.node-5.container_network_receive_pac...   1050
5   container.node-5.container_network_receive_err...   1050
6   container.node-5.container_network_receive_MB....   1050
7   container.node-5.container_network_receive_pac...   1050
8     container.node-5.container_spec_memory_limit_MB    990
9         container.node-5.container_cpu_user_seconds    990
10  container.node-5.container_memory_failures.con...    990
11            container.node-5.container_memory_cache    990
12  container.node-5.container_spec_memory_reserva...    990
13  container.node-5.container_fs_read_seconds./de...    990
14   container.node-5.container_fs_limit_MB./dev/vda1    990
15  container.node-5.container_spec_memory_swap_li...    990
16      container.node-5.container_start_time_seconds    990
17   container.node-5.container_tasks_state.iowaiting    990
18  container.node-5.container_fs_io_current./dev/...    990
19     container.node-5.container_tasks_state.running    990, 'distinct_kpi_name': 1224}, {'rows': 34192, 'min_ts': Timestamp('2022-03-21 01:30:00+0800', tz='Asia/Shanghai'), 'max_ts': Timestamp('2022-03-21 01:59:00+0800', tz='Asia/Shanghai'), 'top_20_cmdb_id':                     cmdb_id  count
0                frontend-0   2664
1                frontend-2   2644
2                frontend-1   2632
3         checkoutservice-2   2356
4         checkoutservice-0   2320
5         checkoutservice-1   2220
6   productcatalogservice-2   1172
7   productcatalogservice-0   1156
8   productcatalogservice-1   1156
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-2    824
15            cartservice-1    808
16            cartservice-0    800
17        currencyservice-2    716
18        currencyservice-1    708
19        currencyservice-0    708, 'top_20_trace_name':                                     trace_name  count
0          trace.from_frontend-0.duration_mean    606
1           trace.from_frontend-0.duration_p95    606
2             trace.from_frontend-0.error_rate    606
3              trace.from_frontend-0.row_count    606
4              trace.from_frontend-2.row_count    601
5             trace.from_frontend-2.error_rate    601
6           trace.from_frontend-2.duration_p95    601
7          trace.from_frontend-2.duration_mean    601
8              trace.from_frontend-1.row_count    598
9             trace.from_frontend-1.error_rate    598
10          trace.from_frontend-1.duration_p95    598
11         trace.from_frontend-1.duration_mean    598
12                    trace.self.duration_mean    479
13                     trace.self.duration_p95    479
14                       trace.self.error_rate    479
15                        trace.self.row_count    479
16  trace.from_checkoutservice-2.duration_mean    465
17   trace.from_checkoutservice-2.duration_p95    465
18     trace.from_checkoutservice-2.error_rate    465
19      trace.from_checkoutservice-2.row_count    465, 'distinct_trace_name': 200}, {'rows': 1796, 'min_ts': Timestamp('2022-03-21 01:30:00+0800', tz='Asia/Shanghai'), 'max_ts': Timestamp('2022-03-21 01:59:00+0800', tz='Asia/Shanghai'), 'top_20_cmdb_id':                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         checkoutservice-0     60
7         checkoutservice-2     60
8         currencyservice-0     60
9         currencyservice-1     60
10        currencyservice-2     60
11         paymentservice-0     60
12               frontend-2     60
13               frontend-1     60
14               frontend-0     60
15        shippingservice-0     60
16  recommendationservice-0     60
17  recommendationservice-1     60
18  recommendationservice-2     60
19         paymentservice-1     60, 'top_20_log_name':           log_name  count
0  log.error_count    898
1    log.row_count    898, 'distinct_log_name': 2}, {'rows': 975, 'min_ts': Timestamp('2022-03-21 01:30:03+0800', tz='Asia/Shanghai'), 'max_ts': Timestamp('2022-03-21 01:59:58+0800', tz='Asia/Shanghai'), 'top_20_cmdb_id':       cmdb_id  count
0  frontend-0    336
1  frontend-2    332
2  frontend-1    307, 'top_20_messages':                                              message  count
0  severity: warning, message: failed to retrieve...    975})```
```

In [ ]:
# In[2]:

import pytz
import pandas as pd

# Reuse existing DataFrames: df_metric, df_trace, df_log, df_error
tz = pytz.timezone('Asia/Shanghai')

# Incident window [inclusive, exclusive)
start = tz.localize(pd.Timestamp('2022-03-21 09:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 10:00:00'))

# Helper to filter and build compact summary
def filter_and_summarize(df, start, end, extra_name_col=None, message_mode=False):
    # Filter using the tz-aware 'ts' column
    df_filt = df[(df['ts'] >= start) & (df['ts'] < end)].copy()
    rows = int(df_filt.shape[0])
    if rows > 0:
        min_ts = df_filt['ts'].min()
        max_ts = df_filt['ts'].max()
        top_cmdb = df_filt['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
    else:
        min_ts = None
        max_ts = None
        top_cmdb = []  # per instruction return empty list when empty
    
    result = {
        'rows': rows,
        'min_ts': min_ts,
        'max_ts': max_ts,
        'top_20_cmdb_id': top_cmdb
    }
    if extra_name_col is not None:
        if rows > 0 and extra_name_col in df_filt.columns:
            vc = df_filt[extra_name_col].value_counts().head(20).rename_axis(extra_name_col).reset_index(name='count')
            distinct = int(df_filt[extra_name_col].nunique())
        else:
            vc = []
            distinct = 0
        result[f'top_20_{extra_name_col}'] = vc
        result[f'distinct_{extra_name_col}'] = distinct
    if message_mode:
        if rows > 0 and 'message' in df_filt.columns:
            msgs = df_filt['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')
        else:
            msgs = []
        result['top_20_messages'] = msgs
    return df_filt, result

# Apply to each DataFrame and store filtered DataFrames and summaries in kernel variables
df_metric_filt, summary_metric_filt = filter_and_summarize(df_metric, start, end, extra_name_col='kpi_name')
df_trace_filt, summary_trace_filt = filter_and_summarize(df_trace, start, end, extra_name_col='trace_name')
df_log_filt, summary_log_filt = filter_and_summarize(df_log, start, end, extra_name_col='log_name')
df_error_filt, summary_error_filt = filter_and_summarize(df_error, start, end, message_mode=True)

# Return only the compact dicts as requested (variables shown as output)
summary_metric_filt, summary_trace_filt, summary_log_filt, summary_error_filt

```
Out[2]:
```
summary_incident_window = (
    "Incident window 2022-03-21 09:30:00+08 (inclusive) to 2022-03-21 10:00:00+08 (exclusive):\n"
    "- All four filtered datasets are empty (0 rows): df_metric_filt, df_trace_filt, df_log_filt, df_error_filt.\n"
    "- No min/max timestamps, no top cmdb_id entries, and no KPI/trace/log/message entries in that window.\n"
    "Conclusion: there is no telemetry data available in the specified incident window for further analysis."
)
summary_incident_window

The original code execution output of IPython Kernel is also provided below for reference:

({'rows': 0, 'min_ts': None, 'max_ts': None, 'top_20_cmdb_id': [], 'top_20_kpi_name': [], 'distinct_kpi_name': 0}, {'rows': 0, 'min_ts': None, 'max_ts': None, 'top_20_cmdb_id': [], 'top_20_trace_name': [], 'distinct_trace_name': 0}, {'rows': 0, 'min_ts': None, 'max_ts': None, 'top_20_cmdb_id': [], 'top_20_log_name': [], 'distinct_log_name': 0}, {'rows': 0, 'min_ts': None, 'max_ts': None, 'top_20_cmdb_id': [], 'top_20_messages': []})```
```

In [ ]:
# In[3]:

import pytz
import pandas as pd

# Reuse existing df_metric and tz-aware 'ts' column in kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for each (cmdb_id, kpi_name) using the entire df_metric
p95_series = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value': 'global_p95'})

# 2) Filter df_metric to the observed telemetry interval 2022-03-21 01:30:00+08 to 2022-03-21 01:59:00+08 (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 01:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 01:59:00'))  # inclusive per instruction
df_metric_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# 3) For each pair, count rows in window with value >= global_P95 (anomaly), earliest anomaly ts, max value in window, and global_P95
# Merge global P95 into windowed data
df_mw_merged = df_metric_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies (>= global_p95). Handle possible NaNs in global_p95 by treating them as non-anomaly.
df_mw_merged['anomaly'] = df_mw_merged['value'] >= df_mw_merged['global_p95']

# Group and summarize
def summarize_group(g):
    anomaly_count = int(g['anomaly'].sum())
    earliest_anomaly_ts = g.loc[g['anomaly'], 'ts'].min() if anomaly_count > 0 else pd.NaT
    max_value_in_window = g['value'].max() if not g['value'].isna().all() else float('nan')
    global_p95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else float('nan')
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_ts': earliest_anomaly_ts,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95
    })

per_pair_summary = df_mw_merged.groupby(['cmdb_id', 'kpi_name'], sort=False).apply(summarize_group).reset_index()

# 4) Prepare compact table: keep only pairs with anomaly_count>0, sort by anomaly_count desc, limit to top 20
anomalies_df = per_pair_summary[per_pair_summary['anomaly_count'] > 0].copy()

if anomalies_df.empty:
    compact_table = []  # per instruction: return empty list if no anomalies
else:
    compact_table = anomalies_df.sort_values(by='anomaly_count', ascending=False).head(20)
    # Keep only requested columns and ensure compact order
    compact_table = compact_table[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'global_p95']]

# Keep intermediate results in kernel variables: p95_df, df_metric_window, per_pair_summary, df_mw_merged
p95_df, df_metric_window, per_pair_summary, compact_table

```
Out[3]:
```
Summary of results (plain English):

- I computed the global 95th percentile (P95) for every (cmdb_id, kpi_name) pair across the entire metric series. The global P95 table is saved as p95_df and contains 4,942 distinct pairs.

- I filtered df_metric to the observed interval 2022-03-21 01:30:00+08 through 2022-03-21 01:59:00+08 (inclusive). The filtered DataFrame df_metric_window contains 147,972 rows (the full available minute-by-minute window).

- For each (cmdb_id, kpi_name) pair I marked values in the window as anomalous when value >= the pair's global P95. The per-pair summary is in per_pair_summary (4,942 rows). The compact anomaly table (compact_table) keeps only pairs with anomaly_count > 0 and is sorted by anomaly_count descending.

- Key findings (top anomalies):
  - Many pairs have anomaly_count = 30 (i.e., anomalous in every minute of the 30-minute window). Examples in the top results include:
    - adservice: app.http.sr (anomaly_count 30, earliest 2022-03-21 01:30:00+08, max 100.0, global_p95 100.0)
    - adservice: app.http.rr (anomaly_count 30, earliest 01:30, max 100.0, global_p95 100.0)
    - adservice: app.http.count (anomaly_count 30, earliest 01:30, max 24.0, global_p95 24.0)
    - shippingservice2-0: a number of container.node-5 related metrics (many with anomaly_count 30, some showing max 0 and global_p95 0)
    - frontend2-0 and other containers also appear with anomaly_count 30 on some container memory metrics.
  - Some pairs show only a few anomaly points (e.g., adservice app.grpc.count had anomaly_count 2 with earliest anomaly at 01:58).

- Interpretation notes:
  - Because anomaly is defined as value >= global P95, some metrics with global_p95 == 0 (or exact equality) will mark zero-valued series as anomalous if their values equal 0.
  - Several service-level KPIs (HTTP/GRPC counts, success rates, etc.) and many container/node-level metrics are among the top anomalous pairs.

- Files/variables saved in the kernel for follow-up:
  - p95_df (global P95 per pair)
  - df_metric_window (filtered window)
  - df_mw_merged (windowed records merged with global P95 and anomaly flag)
  - per_pair_summary (per-pair anomaly counts and stats)
  - compact_table (top anomalous pairs, limited to 20 rows in the displayed compact table)

If you want, I can show the top 20 rows of compact_table in tabular form or give the exact count of pairs with anomaly_count > 0.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name    global_p95
0              adservice                                     app.grpc.count    149.200000
1              adservice                                       app.grpc.mrt      0.002892
2              adservice                                        app.grpc.rr    100.000000
3              adservice                                        app.grpc.sr     76.274384
4              adservice                                     app.http.count     24.000000
...                  ...                                                ...           ...
4937  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     15.500000
4938  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  43912.500000
4939  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.000000
4940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     14.275000
4941  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1420.362500

[4942 rows x 3 columns],          timestamp             cmdb_id                                           kpi_name         value                        ts
0       1647826200           adservice                                     app.grpc.count     96.000000 2022-03-21 01:30:00+08:00
1       1647826200           adservice                                       app.grpc.mrt      0.002446 2022-03-21 01:30:00+08:00
2       1647826200           adservice                                        app.grpc.rr    100.000000 2022-03-21 01:30:00+08:00
3       1647826200           adservice                                        app.grpc.sr     71.875000 2022-03-21 01:30:00+08:00
4       1647826200           adservice                                     app.http.count     24.000000 2022-03-21 01:30:00+08:00
...            ...                 ...                                                ...           ...                       ...
147967  1647827940  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      7.000000 2022-03-21 01:59:00+08:00
147968  1647827940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  14150.000000 2022-03-21 01:59:00+08:00
147969  1647827940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.000000 2022-03-21 01:59:00+08:00
147970  1647827940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      5.000000 2022-03-21 01:59:00+08:00
147971  1647827940  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    497.500000 2022-03-21 01:59:00+08:00

[147972 rows x 5 columns],         cmdb_id              kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window    global_p95
0     adservice        app.grpc.count              2 2022-03-21 01:58:00+08:00           152.000000    149.200000
1     adservice          app.grpc.mrt              2 2022-03-21 01:42:00+08:00             0.003407      0.002892
2     adservice           app.grpc.rr             30 2022-03-21 01:30:00+08:00           100.000000    100.000000
3     adservice           app.grpc.sr              2 2022-03-21 01:33:00+08:00            77.142857     76.274384
4     adservice        app.http.count             30 2022-03-21 01:30:00+08:00            24.000000     24.000000
...         ...                   ...            ...                       ...                  ...           ...
4937     node-3     system.mem.usable              1 2022-03-21 01:53:00+08:00         14178.000000  14175.750000
4938     node-4  system.mem.real.used              1 2022-03-21 01:53:00+08:00         15352.000000  15351.000000
4939     node-4     system.mem.usable              2 2022-03-21 01:33:00+08:00         14932.000000  14932.000000
4940     node-6  system.mem.real.used              1 2022-03-21 01:58:00+08:00          9159.000000   9157.750000
4941     node-6     system.mem.usable              1 2022-03-21 01:33:00+08:00         21966.000000  21964.750000

[4942 rows x 6 columns],                  cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window    global_p95
7              adservice                                        app.http.sr             30 2022-03-21 01:30:00+08:00           100.000000    100.000000
4891  shippingservice2-0  container.node-5.container_network_receive_err...             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4893  shippingservice2-0  container.node-5.container_network_receive_pac...             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4895  shippingservice2-0  container.node-5.container_network_transmit_er...             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
6              adservice                                        app.http.rr             30 2022-03-21 01:30:00+08:00           100.000000    100.000000
4              adservice                                     app.http.count             30 2022-03-21 01:30:00+08:00            24.000000     24.000000
4885  shippingservice2-0     container.node-5.container_memory_max_usage_MB             30 2022-03-21 01:30:00+08:00           128.011719    128.011719
4887  shippingservice2-0             container.node-5.container_memory_swap             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4872  shippingservice2-0  container.node-5.container_fs_write_seconds./d...             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4873  shippingservice2-0      container.node-5.container_fs_writes./dev/vda             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4874  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4875  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4876  shippingservice2-0  container.node-5.container_fs_writes_merged./d...             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
2              adservice                                        app.grpc.rr             30 2022-03-21 01:30:00+08:00           100.000000    100.000000
3043         frontend2-0     container.node-6.container_memory_max_usage_MB             30 2022-03-21 01:30:00+08:00           125.523438    125.523438
3045         frontend2-0             container.node-6.container_memory_swap             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4927  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4912  shippingservice2-0             container.node-5.container_threads_max             30 2022-03-21 01:30:00+08:00             0.000000      0.000000
4913  shippingservice2-0  container.node-5.container_ulimits_soft.max_op...             30 2022-03-21 01:30:00+08:00         65536.000000  65536.000000
4904  shippingservice2-0  container.node-5.container_spec_memory_swap_li...             30 2022-03-21 01:30:00+08:00           128.000000    128.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: per_pair_summary, p95_df, df_metric_window, df_mw_merged, compact_table

# 1) Define allowed root-cause component set exactly as in the spec
nodes = [f"node-{i}" for i in range(1, 7)]
services = [
    "frontend", "shippingservice", "checkoutservice", "currencyservice",
    "adservice", "emailservice", "cartservice", "productcatalogservice",
    "recommendationservice", "paymentservice"
]
pods = [
    "frontend-0", "frontend-1", "frontend-2", "frontend2-0",
    "shippingservice-0", "shippingservice-1", "shippingservice-2", "shippingservice2-0",
    "checkoutservice-0", "checkoutservice-1", "checkoutservice-2", "checkoutservice2-0",
    "currencyservice-0", "currencyservice-1", "currencyservice-2", "currencyservice2-0",
    "adservice-0", "adservice-1", "adservice-2", "adservice2-0",
    "emailservice-0", "emailservice-1", "emailservice-2", "emailservice2-0",
    "cartservice-0", "cartservice-1", "cartservice-2", "cartservice2-0",
    "productcatalogservice-0", "productcatalogservice-1", "productcatalogservice-2", "productcatalogservice2-0",
    "recommendationservice-0", "recommendationservice-1", "recommendationservice-2", "recommendationservice2-0",
    "paymentservice-0", "paymentservice-1", "paymentservice-2", "paymentservice2-0"
]

allowed_components = nodes + services + pods

# 2) Filter per_pair_summary to anomaly_count > 0 and cmdb_id in allowed set
anomalous_pairs_allowed = per_pair_summary[
    (per_pair_summary['anomaly_count'] > 0) &
    (per_pair_summary['cmdb_id'].isin(allowed_components))
].copy()

# 3) Compute required aggregations per cmdb_id
# Compute per-pair breaches first
anomalous_pairs_allowed['abs_breach'] = anomalous_pairs_allowed['max_value_in_window'] - anomalous_pairs_allowed['global_p95']

# Relative breach rules:
# - if global_p95 > 0: (max - p95) / p95
# - if global_p95 == 0 and max_value_in_window > 0: large number (1e6)
# - else 0
def compute_rel(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(gp):
        return 0.0
    if gp > 0:
        return (mv - gp) / gp
    else:
        return 1e6 if (mv > 0) else 0.0

anomalous_pairs_allowed['rel_breach'] = anomalous_pairs_allowed.apply(compute_rel, axis=1)

# Now group by cmdb_id
grouped = anomalous_pairs_allowed.groupby('cmdb_id', sort=False)

per_component_summary = grouped.agg(
    total_anomaly_count = ('anomaly_count', 'sum'),
    num_distinct_kpis_anomalous = ('kpi_name', 'nunique'),
    earliest_anomaly_ts = ('earliest_anomaly_ts', 'min'),
    max_abs_breach = ('abs_breach', 'max'),
    max_relative_breach = ('rel_breach', 'max')
).reset_index()

# 4) Sort by total_anomaly_count desc, then by max_relative_breach desc, limit to top 20
per_component_summary = per_component_summary.sort_values(
    by=['total_anomaly_count', 'max_relative_breach'],
    ascending=[False, False]
).head(20)

# Keep intermediate variables in kernel for follow-up
anomalous_pairs_allowed, per_component_summary

```
Out[4]:
```
```python
# Summarize the aggregated per-component anomaly table (per_component_summary) in plain English.
# The summary string is returned as the final variable (no raw tables printed).

def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    try:
        return ts.strftime('%Y-%m-%d %H:%M:%S %z')
    except Exception:
        return str(ts)

rows = per_component_summary.shape[0]
lines = []
lines.append(f"Aggregated anomaly summary for allowed components (top {rows} shown):")
for i, r in per_component_summary.reset_index(drop=True).iterrows():
    cid = r['cmdb_id']
    total = int(r['total_anomaly_count'])
    num_kpis = int(r['num_distinct_kpis_anomalous'])
    earliest = fmt_ts(r['earliest_anomaly_ts'])
    abs_b = r['max_abs_breach']
    rel_b = r['max_relative_breach']
    # compact numeric formatting
    abs_b_s = f"{abs_b:.6g}"
    rel_b_s = ("{:.6g}".format(rel_b)) if rel_b < 1e6 else ">>1e6 (p95=0 & value>0)"
    lines.append(f"{i+1}. {cid}: total_anomaly_count={total}, num_kpis_anomalous={num_kpis}, earliest_anomaly_ts={earliest}, max_abs_breach={abs_b_s}, max_relative_breach={rel_b_s}")

lines.append("")
lines.append("Notable observations:")
lines.append("- adservice has the highest total anomalies (8957) across many KPIs (341 distinct), earliest at 2022-03-21 01:30, and large absolute breaches.")
lines.append("- Multiple frontend pods (frontend2-0, frontend-1, frontend-0, frontend-2) also show very high anomaly counts.")
lines.append("- Some components show extremely large relative breach values (marked as >>1e6) because global P95 == 0 while observed max > 0; these represent zero-denominator cases handled per rules.")
lines.append("- Earliest anomalies for top components occur at 2022-03-21 01:30:00+08, indicating anomalies throughout the observed window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id              kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window    global_p95  abs_breach  rel_breach
0     adservice        app.grpc.count              2 2022-03-21 01:58:00+08:00           152.000000    149.200000    2.800000    0.018767
1     adservice          app.grpc.mrt              2 2022-03-21 01:42:00+08:00             0.003407      0.002892    0.000515    0.177999
2     adservice           app.grpc.rr             30 2022-03-21 01:30:00+08:00           100.000000    100.000000    0.000000    0.000000
3     adservice           app.grpc.sr              2 2022-03-21 01:33:00+08:00            77.142857     76.274384    0.868474    0.011386
4     adservice        app.http.count             30 2022-03-21 01:30:00+08:00            24.000000     24.000000    0.000000    0.000000
...         ...                   ...            ...                       ...                  ...           ...         ...         ...
4937     node-3     system.mem.usable              1 2022-03-21 01:53:00+08:00         14178.000000  14175.750000    2.250000    0.000159
4938     node-4  system.mem.real.used              1 2022-03-21 01:53:00+08:00         15352.000000  15351.000000    1.000000    0.000065
4939     node-4     system.mem.usable              2 2022-03-21 01:33:00+08:00         14932.000000  14932.000000    0.000000    0.000000
4940     node-6  system.mem.real.used              1 2022-03-21 01:58:00+08:00          9159.000000   9157.750000    1.250000    0.000136
4941     node-6     system.mem.usable              1 2022-03-21 01:33:00+08:00         21966.000000  21964.750000    1.250000    0.000057

[4585 rows x 8 columns],                      cmdb_id  total_anomaly_count  num_distinct_kpis_anomalous       earliest_anomaly_ts  max_abs_breach  max_relative_breach
0                  adservice                 8957                          341 2022-03-21 01:30:00+08:00    2.466250e+08             1.545455
29               frontend2-0                 3304                          176 2022-03-21 01:30:00+08:00    1.100800e+05             2.043011
27                frontend-1                 3004                          168 2022-03-21 01:30:00+08:00    1.368064e+05             0.445313
26                frontend-0                 2828                          164 2022-03-21 01:30:00+08:00    2.185216e+05             1.709441
28                frontend-2                 2552                          154 2022-03-21 01:30:00+08:00    1.065984e+05       1000000.000000
43   productcatalogservice-1                 2203                          116 2022-03-21 01:30:00+08:00    9.625600e+04             1.000000
11         checkoutservice-0                 2175                          130 2022-03-21 01:30:00+08:00    2.781184e+05             0.598956
42   productcatalogservice-0                 2167                          116 2022-03-21 01:30:00+08:00    2.549760e+04             0.420262
7              cartservice-1                 2166                          100 2022-03-21 01:30:00+08:00    2.716672e+05             0.565196
44   productcatalogservice-2                 2145                          116 2022-03-21 01:30:00+08:00    8.284160e+04             0.290323
4               adservice2-0                 2124                          102 2022-03-21 01:30:00+08:00    1.310720e+05             6.121322
6              cartservice-0                 2049                           96 2022-03-21 01:30:00+08:00    1.943552e+05             0.592392
2                adservice-1                 1956                           86 2022-03-21 01:30:00+08:00    2.143750e+03            10.722991
3                adservice-2                 1951                           86 2022-03-21 01:30:00+08:00    2.537500e+03             0.600385
13         checkoutservice-2                 1901                          122 2022-03-21 01:30:00+08:00    1.429504e+05             9.010477
9             cartservice2-0                 1887                           96 2022-03-21 01:30:00+08:00    2.940928e+05       1000000.000000
14        checkoutservice2-0                 1885                          118 2022-03-21 01:30:00+08:00    2.340864e+05       1000000.000000
45  productcatalogservice2-0                 1872                           96 2022-03-21 01:30:00+08:00    1.593344e+05             0.312009
50  recommendationservice2-0                 1830                           84 2022-03-21 01:30:00+08:00    1.023750e+04             3.504730
48   recommendationservice-1                 1816                           84 2022-03-21 01:30:00+08:00    4.112500e+03             3.926702)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing df_trace from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for each (cmdb_id, trace_name) using the entire df_trace
trace_p95_df = df_trace.groupby(['cmdb_id', 'trace_name'], sort=False)['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})

# 2) Filter df_trace to the observed telemetry window 2022-03-21 01:30:00+08 through 2022-03-21 01:59:00+08 (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 01:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 01:59:00'))
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# 3) Merge global P95 into windowed trace data and mark anomalies (value >= global_p95)
df_trace_mw_merged = df_trace_window.merge(trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')
df_trace_mw_merged['anomaly'] = df_trace_mw_merged['value'] >= df_trace_mw_merged['global_p95']

# Group and summarize per (cmdb_id, trace_name)
def summarize_trace_group(g):
    anomaly_count = int(g['anomaly'].sum())
    earliest_anomaly_ts = g.loc[g['anomaly'], 'ts'].min() if anomaly_count > 0 else pd.NaT
    max_value_in_window = g['value'].max() if not g['value'].isna().all() else float('nan')
    global_p95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else float('nan')
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_ts': earliest_anomaly_ts,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95
    })

per_pair_trace_summary = df_trace_mw_merged.groupby(['cmdb_id', 'trace_name'], sort=False).apply(summarize_trace_group).reset_index()

# 4) Prepare compact table: keep only pairs with anomaly_count > 0, sort by anomaly_count desc, limit to top 20
compact_trace_table = per_pair_trace_summary[per_pair_trace_summary['anomaly_count'] > 0].copy()
if compact_trace_table.empty:
    compact_trace_table = []  # per instruction: return empty list if no anomalies
else:
    compact_trace_table = compact_trace_table.sort_values(by='anomaly_count', ascending=False).head(20)
    compact_trace_table = compact_trace_table[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'global_p95']]

# Keep trace_p95_df, df_trace_window, per_pair_trace_summary, df_trace_mw_merged in kernel for follow-up
compact_trace_table

```
Out[5]:
```
Summary of trace anomaly check (plain English):

- I computed global P95 per (cmdb_id, trace_name) and inspected the 01:30–01:59+08 window.
- Many trace pairs are flagged as anomalous for every minute in the window (anomaly_count = 30), with earliest anomaly timestamp 2022-03-21 01:30:00+08.
- In the top examples shown (first 20 rows), all anomalous pairs have max_value_in_window = 0.0 and global_p95 = 0.0. Examples include:
  - adservice-0 — trace.from_frontend-0.error_rate (30 anomalies)
  - adservice-1 — trace.from_frontend-2.error_rate (30)
  - frontend-0 — trace.to_cartservice-2.error_rate (30)
  - cartservice-1 — trace.from_frontend-0.error_rate (30)
  - productcatalogservice-0 — trace.from_frontend-1.error_rate (30)
  - (and others; 20 pairs shown)
- Interpretation: because anomaly was defined as value >= global P95, pairs whose global P95 equals 0 mark zero-valued records as anomalies (value == 0 => anomaly). This makes the anomaly counts reflect zero-valued metrics rather than clear positive breaches.
- Saved kernel variables for follow-up: trace_p95_df (global P95), df_trace_window (windowed trace data), per_pair_trace_summary and df_trace_mw_merged (merged window data with anomaly flags).
- Suggested next step: if you want meaningful trace anomalies, exclude pairs with global_p95 == 0 or use a different rule (e.g., value > global_p95, or require absolute/relative thresholds) — tell me which rule to apply and I will recompute.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  anomaly_count       earliest_anomaly_ts  max_value_in_window  global_p95
2                adservice-0                trace.from_frontend-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
22               adservice-1                trace.from_frontend-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
494               frontend-0               trace.to_cartservice-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
78             cartservice-1                trace.from_frontend-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
10               adservice-0                trace.from_frontend-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
486               frontend-0               trace.to_cartservice-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
490               frontend-0               trace.to_cartservice-1.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
530               frontend-0     trace.to_productcatalogservice-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
518               frontend-0           trace.to_currencyservice-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
38              adservice2-0               trace.from_frontend2-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
794  productcatalogservice-0                trace.from_frontend-1.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
846  productcatalogservice-1   trace.from_recommendationservice-2.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
842  productcatalogservice-1   trace.from_recommendationservice-1.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
346       checkoutservice2-0              trace.to_cartservice2-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
782        paymentservice2-0        trace.from_checkoutservice2-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
390        currencyservice-1         trace.from_checkoutservice-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
382        currencyservice-0                trace.from_frontend-1.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
790  productcatalogservice-0                trace.from_frontend-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
758              frontend2-0          trace.to_shippingservice2-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0
814  productcatalogservice-0  trace.from_recommendationservice2-0.error_rate             30 2022-03-21 01:30:00+08:00                  0.0         0.0```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse kernel variables: per_component_summary, df_error
tz = pytz.timezone('Asia/Shanghai')

# 1) Component severity table: filter to total_anomaly_count>0 and sort by max_relative_breach desc then total_anomaly_count desc
severity_table = (
    per_component_summary[per_component_summary['total_anomaly_count'] > 0]
    .copy()
    .sort_values(by=['max_relative_breach', 'total_anomaly_count'], ascending=[False, False])
)

# Keep only requested columns and limit to top 10
severity_table = severity_table[
    ['cmdb_id', 'total_anomaly_count', 'num_distinct_kpis_anomalous', 'earliest_anomaly_ts', 'max_abs_breach', 'max_relative_breach']
].head(10).reset_index(drop=True)

# 2) Error log summary for top 5 components from severity_table
top5_components = severity_table['cmdb_id'].head(5).tolist()

# Define observed telemetry window (inclusive) in UTC+8
start = tz.localize(pd.Timestamp('2022-03-21 01:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 01:59:00'))

# Prepare list of summaries
error_summaries = []
for comp in top5_components:
    df_comp_err = df_error[(df_error['cmdb_id'] == comp) & (df_error['ts'] >= start) & (df_error['ts'] <= end)].copy()
    error_rows_count = int(df_comp_err.shape[0])
    earliest_error_ts = df_comp_err['ts'].min() if error_rows_count > 0 else pd.NaT
    if error_rows_count > 0 and 'message' in df_comp_err.columns:
        top_messages = df_comp_err['message'].value_counts().head(5).rename_axis('message').reset_index(name='count')
    else:
        top_messages = pd.DataFrame(columns=['message', 'count'])
    error_summaries.append({
        'cmdb_id': comp,
        'error_rows_count': error_rows_count,
        'earliest_error_ts': earliest_error_ts if not pd.isna(earliest_error_ts) else None,
        'top_messages': top_messages  # keep as DataFrame for compact inspection if needed
    })

# Convert error_summaries to a compact DataFrame for display (top_messages kept as object)
error_summary_df = pd.DataFrame([
    {
        'cmdb_id': e['cmdb_id'],
        'error_rows_count': e['error_rows_count'],
        'earliest_error_ts': e['earliest_error_ts'],
        # represent top_messages as list of "message (count)" strings for compactness
        'top_messages': (e['top_messages'].apply(lambda r: f"{r['message']} ({r['count']})", axis=1).tolist()
                         if not e['top_messages'].empty else [])
    } for e in error_summaries
])

# Keep intermediate results in kernel: severity_table, error_summary_df, error_summaries, top5_components
severity_table, error_summary_df

```
Out[6]:
```
```python
# Plain-English summary string based on the kernel results (severity_table and error_summary_df).
summary = (
    "Component severity summary (top 10 by relative breach then anomaly count):\n"
    "1) frontend-2: total_anomaly_count=2552, num_kpis_anomalous=154, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=106598.40, max_relative_breach=1e6 (zero-denominator case)\n"
    "2) cartservice2-0: total_anomaly_count=1887, num_kpis_anomalous=96, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=294092.80, max_relative_breach=1e6\n"
    "3) checkoutservice2-0: total_anomaly_count=1885, num_kpis_anomalous=118, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=234086.40, max_relative_breach=1e6\n"
    "4) adservice-1: total_anomaly_count=1956, num_kpis_anomalous=86, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=2143.75, max_relative_breach≈10.72\n"
    "5) checkoutservice-2: total_anomaly_count=1901, num_kpis_anomalous=122, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=142950.40, max_relative_breach≈9.01\n"
    "6) adservice2-0: total_anomaly_count=2124, num_kpis_anomalous=102, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=131072.00, max_relative_breach≈6.12\n"
    "7) recommendationservice-1: total_anomaly_count=1816, num_kpis_anomalous=84, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=4112.50, max_relative_breach≈3.93\n"
    "8) recommendationservice2-0: total_anomaly_count=1830, num_kpis_anomalous=84, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=10237.50, max_relative_breach≈3.50\n"
    "9) frontend2-0: total_anomaly_count=3304, num_kpis_anomalous=176, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=110080.00, max_relative_breach≈2.04\n"
    "10) frontend-0: total_anomaly_count=2828, num_kpis_anomalous=164, earliest_anomaly_ts=2022-03-21 01:30:00+08, "
    "max_abs_breach=218521.60, max_relative_breach≈1.71\n\n"
    "Error log summary for top 5 components (window 2022-03-21 01:30:00+08 through 2022-03-21 01:59:00+08):\n"
    "- frontend-2: error_rows_count=320, earliest_error_ts=2022-03-21 01:30:06+08, top messages include a repeated warning "
    "starting with 'severity: warning, message: failed to retriev...'.\n"
    "- cartservice2-0: error_rows_count=0, no errors in the window.\n"
    "- checkoutservice2-0: error_rows_count=0, no errors in the window.\n"
    "- adservice-1: error_rows_count=0, no errors in the window.\n"
    "- checkoutservice-2: error_rows_count=0, no errors in the window.\n\n"
    "Notes/interpretation:\n"
    "- Several components show extremely large relative-breach values (displayed as 1e6) because their global P95 == 0 while observed max > 0; these are zero-denominator cases handled per rules and should be interpreted cautiously.\n"
    "- frontend-2 is the only top candidate with many error-log entries in the window (320 warnings); the other top components had no error log rows in the same interval.\n"
    "- Next steps: investigate frontend-2's warnings, and inspect the high-breach KPIs for the other components (pay attention to P95==0 cases where the anomaly definition may be producing false positives).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  total_anomaly_count  num_distinct_kpis_anomalous       earliest_anomaly_ts  max_abs_breach  max_relative_breach
0                frontend-2                 2552                          154 2022-03-21 01:30:00+08:00       106598.40       1000000.000000
1            cartservice2-0                 1887                           96 2022-03-21 01:30:00+08:00       294092.80       1000000.000000
2        checkoutservice2-0                 1885                          118 2022-03-21 01:30:00+08:00       234086.40       1000000.000000
3               adservice-1                 1956                           86 2022-03-21 01:30:00+08:00         2143.75            10.722991
4         checkoutservice-2                 1901                          122 2022-03-21 01:30:00+08:00       142950.40             9.010477
5              adservice2-0                 2124                          102 2022-03-21 01:30:00+08:00       131072.00             6.121322
6   recommendationservice-1                 1816                           84 2022-03-21 01:30:00+08:00         4112.50             3.926702
7  recommendationservice2-0                 1830                           84 2022-03-21 01:30:00+08:00        10237.50             3.504730
8               frontend2-0                 3304                          176 2022-03-21 01:30:00+08:00       110080.00             2.043011
9                frontend-0                 2828                          164 2022-03-21 01:30:00+08:00       218521.60             1.709441,               cmdb_id  error_rows_count         earliest_error_ts                                       top_messages
0          frontend-2               320 2022-03-21 01:30:06+08:00  [severity: warning, message: failed to retriev...
1      cartservice2-0                 0                       NaT                                                 []
2  checkoutservice2-0                 0                       NaT                                                 []
3         adservice-1                 0                       NaT                                                 []
4   checkoutservice-2                 0                       NaT                                                 [])```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables: per_pair_summary, df_error
tz = pytz.timezone('Asia/Shanghai')

# Observed window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 01:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 01:59:00'))

# 1) frontend-2 KPI anomaly table
df_frontend2_pairs = per_pair_summary[
    (per_pair_summary['cmdb_id'] == 'frontend-2') &
    (per_pair_summary['anomaly_count'] > 0)
].copy()

if df_frontend2_pairs.empty:
    frontend2_kpi_table = []  # per instruction return empty list if no anomalies
else:
    # Compute abs_breach
    df_frontend2_pairs['abs_breach'] = df_frontend2_pairs['max_value_in_window'] - df_frontend2_pairs['global_p95']
    # Compute numeric rel_breach for sorting:
    def rel_numeric(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return 0.0
        if gp > 0:
            return (mv - gp) / gp
        else:
            # gp == 0
            return 1e12 if (mv > 0) else 0.0
    df_frontend2_pairs['rel_breach_numeric'] = df_frontend2_pairs.apply(rel_numeric, axis=1)
    # Create display column for rel_breach per rules
    def rel_display(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return 0.0
        if gp == 0:
            return '>>1e6 (p95=0 & value>0)' if (mv > 0) else 0.0
        else:
            # format to 6 significant digits for compactness
            val = (mv - gp) / gp
            return float(f"{val:.6g}")
    df_frontend2_pairs['rel_breach'] = df_frontend2_pairs.apply(rel_display, axis=1)
    # Sort by rel_breach_numeric desc, then anomaly_count desc
    df_frontend2_pairs = df_frontend2_pairs.sort_values(by=['rel_breach_numeric', 'anomaly_count'], ascending=[False, False])
    # Select and rename columns as requested, limit to top 20
    frontend2_kpi_table = df_frontend2_pairs.loc[:, [
        'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window', 'global_p95', 'abs_breach', 'rel_breach'
    ]].head(20).reset_index(drop=True)

# Keep df_frontend2_pairs and frontend2_kpi_table in kernel for follow-up

# 2) frontend-2 error-log evidence
df_frontend2_errors = df_error[
    (df_error['cmdb_id'] == 'frontend-2') &
    (df_error['ts'] >= start) &
    (df_error['ts'] <= end)
].copy()

error_rows_count = int(df_frontend2_errors.shape[0])
earliest_error_ts = df_frontend2_errors['ts'].min() if error_rows_count > 0 else None

# Top up to 10 distinct messages with counts ordered by count desc
if error_rows_count > 0 and 'message' in df_frontend2_errors.columns:
    top_msgs_df = df_frontend2_errors['message'].value_counts().head(10).rename_axis('message').reset_index(name='count')
    top_messages = list(top_msgs_df.itertuples(index=False, name=None))  # list of (message, count)
else:
    top_messages = []

# Examples: up to 5 example rows as tuples (timestamp, message) ordered by timestamp asc
if error_rows_count > 0:
    examples_df = df_frontend2_errors.sort_values('ts').head(5)[['ts', 'message']].copy()
    # format timestamp strings for compactness
    examples = [(ts.isoformat(), msg) for ts, msg in zip(examples_df['ts'], examples_df['message'])]
else:
    examples = []

frontend2_error_summary = {
    'cmdb_id': 'frontend-2',
    'error_rows_count': error_rows_count,
    'earliest_error_ts': earliest_error_ts,
    'top_messages': top_messages,
    'examples': examples
}

# Return only the two compact outputs (DataFrame and dict). Keep intermediates in kernel.
frontend2_kpi_table, frontend2_error_summary

```
Out[8]:
```
```python
# Compose a concise plain-English summary for frontend-2 using existing kernel variables:
# frontend2_kpi_table (DataFrame or empty list) and frontend2_error_summary (dict).
# The summary string is returned as the final variable.

import pandas as pd

# Ensure variables exist in kernel (they were created earlier)
try:
    table = frontend2_kpi_table
    err = frontend2_error_summary
except NameError:
    table = []
    err = {}

def fmt_ts(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return "None"
    try:
        return pd.Timestamp(val).strftime('%Y-03-%d %H:%M:%S %z') if hasattr(val, 'tzinfo') else pd.Timestamp(val).isoformat()
    except Exception:
        return str(val)

if isinstance(table, list) and len(table) == 0:
    summary = ("frontend-2 summary:\n"
               "- No KPI anomalies found for frontend-2 in the observed window.\n"
               f"- Error log: {err.get('error_rows_count', 0)} rows; earliest_error_ts={fmt_ts(err.get('earliest_error_ts'))}.\n")
else:
    # table is a DataFrame: produce top 3 KPI highlights
    df = table.copy()
    n_kpis = int(df.shape[0])
    top_n = min(3, n_kpis)
    highlights = []
    for i, row in df.head(top_n).iterrows():
        k = row['kpi_name']
        ac = int(row['anomaly_count'])
        ets = fmt_ts(row['earliest_anomaly_ts'])
        mv = row['max_value_in_window']
        gp = row['global_p95']
        abs_b = row['abs_breach']
        rel = row['rel_breach']
        highlights.append(f"- {k}: anomaly_count={ac}, earliest={ets}, max={mv}, p95={gp}, abs_breach={abs_b}, rel_breach={rel}")

    # Error summary
    err_rows = int(err.get('error_rows_count', 0))
    earliest_err = fmt_ts(err.get('earliest_error_ts'))
    top_msgs = err.get('top_messages', [])
    top_msgs_str = "; ".join([f"\"{m}\" ({c})" for m, c in top_msgs]) if top_msgs else "[]"
    examples = err.get('examples', [])

    summary_lines = [
        "frontend-2 summary (observed window 2022-03-21 01:30–01:59+08):",
        f"- KPI anomaly pairs found: {n_kpis}. Top {top_n} KPI highlights:",
    ]
    summary_lines.extend(highlights)
    summary_lines.append(f"- Error logs for frontend-2 in window: error_rows_count={err_rows}, earliest_error_ts={earliest_err}")
    summary_lines.append(f"- Top messages (up to 10): {top_msgs_str}")
    if examples:
        example_lines = ", ".join([f"({ts}, {msg})" for ts, msg in examples])
        summary_lines.append(f"- Example rows (up to 5 earliest): {example_lines}")
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window    global_p95   abs_breach               rel_breach
0   container.node-5.container_cpu_cfs_throttled_s...             30 2022-03-21 01:30:00+08:00             0.001325      0.000000     0.001325  >>1e6 (p95=0 & value>0)
1   mesh.source.frontend.shippingservice.istio_req...              2 2022-03-21 01:33:00+08:00             0.053500      0.034386     0.019114                 0.555854
2   mesh.source.frontend.shippingservice.istio_req...              2 2022-03-21 01:51:00+08:00            22.000000     16.275000     5.725000                 0.351767
3   mesh.source.frontend.shippingservice.istio_req...              2 2022-03-21 01:51:00+08:00            22.000000     16.275000     5.725000                 0.351767
4   mesh.source.frontend.shippingservice.istio_res...              2 2022-03-21 01:51:00+08:00            22.000000     16.275000     5.725000                 0.351767
5   container.node-5.container_network_receive_MB....              2 2022-03-21 01:42:00+08:00             0.456587      0.361859     0.094728                 0.261782
6   mesh.source.frontend.shippingservice.istio_res...              2 2022-03-21 01:51:00+08:00          3332.500000   2720.250000   612.250000                 0.225071
7   mesh.source.frontend.shippingservice.istio_req...              2 2022-03-21 01:51:00+08:00         33325.000000  27202.500000  6122.500000                 0.225071
8   mesh.destination.unknown.frontend.istio_reques...              2 2022-03-21 01:49:00+08:00             0.073550      0.060335     0.013215                 0.219027
9   mesh.source.frontend.checkoutservice.istio_req...              2 2022-03-21 01:53:00+08:00          7425.000000   6228.750000  1196.250000                 0.192053
10  mesh.source.frontend.checkoutservice.istio_req...              2 2022-03-21 01:53:00+08:00             4.500000      3.775000     0.725000                 0.192053
11  mesh.source.frontend.checkoutservice.istio_req...              2 2022-03-21 01:53:00+08:00             4.500000      3.775000     0.725000                 0.192053
12  mesh.source.frontend.checkoutservice.istio_res...              2 2022-03-21 01:53:00+08:00             4.500000      3.775000     0.725000                 0.192053
13  mesh.source.frontend.checkoutservice.istio_res...              2 2022-03-21 01:53:00+08:00          1547.500000   1307.125000   240.375000                 0.183896
14  mesh.source.frontend.jaeger-collector.istio_re...              2 2022-03-21 01:44:00+08:00             0.008400      0.007278     0.001123                 0.154243
15        container.node-5.container_cpu_user_seconds              4 2022-03-21 01:51:00+08:00             0.255000      0.225000     0.030000                 0.133333
16         container.node-5.container_cpu_cfs_periods              3 2022-03-21 01:53:00+08:00           164.500000    147.000000    17.500000                 0.119048
17      container.node-5.container_cpu_system_seconds              2 2022-03-21 01:53:00+08:00             0.145000      0.132750     0.012250                 0.092279
18  mesh.destination.unknown.frontend.istio_reques...              2 2022-03-21 01:51:00+08:00          5487.500000   5060.750000   426.750000                 0.084325
19  mesh.destination.unknown.frontend.istio_reques...              4 2022-03-21 01:51:00+08:00            14.000000     13.000000     1.000000                 0.076923, {'cmdb_id': 'frontend-2', 'error_rows_count': 320, 'earliest_error_ts': Timestamp('2022-03-21 01:30:06+0800', tz='Asia/Shanghai'), 'top_messages': [('severity: warning, message: failed to retrieve ads', 320)], 'examples': [('2022-03-21T01:30:06+08:00', 'severity: warning, message: failed to retrieve ads'), ('2022-03-21T01:30:13+08:00', 'severity: warning, message: failed to retrieve ads'), ('2022-03-21T01:30:16+08:00', 'severity: warning, message: failed to retrieve ads'), ('2022-03-21T01:30:19+08:00', 'severity: warning, message: failed to retrieve ads'), ('2022-03-21T01:30:26+08:00', 'severity: warning, message: failed to retrieve ads')]})```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables: per_pair_summary, p95_df, df_error
tz = pytz.timezone('Asia/Shanghai')

# Observed window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 01:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 01:59:00'))

# 1) Per-KPI anomaly summary for cmdb_id == 'adservice'
ad_pairs = per_pair_summary[per_pair_summary['cmdb_id'] == 'adservice'].copy()

# Keep only anomaly_count > 0 per earlier logic (anomalies within window)
ad_pairs = ad_pairs[ad_pairs['anomaly_count'] > 0].copy()

if not ad_pairs.empty:
    # compute abs_breach
    ad_pairs['abs_breach'] = ad_pairs['max_value_in_window'] - ad_pairs['global_p95']
    # numeric rel breach for sorting: handle p95==0 case
    def rel_num(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return 0.0
        if gp > 0:
            return (mv - gp) / gp
        else:
            # gp == 0
            return 1e12 if (mv > 0) else 0.0
    ad_pairs['rel_breach_numeric'] = ad_pairs.apply(rel_num, axis=1)
    # display rel_breach per rules
    def rel_display(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return 0.0
        if gp == 0:
            return '>>1e6 (p95=0 & value>0)' if (mv > 0) else 0.0
        else:
            return float(f"{((mv - gp) / gp):.6g}")
    ad_pairs['rel_breach'] = ad_pairs.apply(rel_display, axis=1)
    # sort by rel_breach_numeric desc then anomaly_count desc
    ad_pairs = ad_pairs.sort_values(by=['rel_breach_numeric', 'anomaly_count'], ascending=[False, False])
    # select requested columns and limit to top 20
    adservice_kpi_table = ad_pairs.loc[:, [
        'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window',
        'global_p95', 'abs_breach', 'rel_breach'
    ]].head(20).reset_index(drop=True)
else:
    adservice_kpi_table = pd.DataFrame(columns=[
        'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_value_in_window',
        'global_p95', 'abs_breach', 'rel_breach'
    ])

# 2) Error log summary for adservice in the same observed window
df_ad_errors = df_error[
    (df_error['cmdb_id'] == 'adservice') &
    (df_error['ts'] >= start) &
    (df_error['ts'] <= end)
].copy()

error_rows_count = int(df_ad_errors.shape[0])
earliest_error_ts = df_ad_errors['ts'].min() if error_rows_count > 0 else None

adservice_error_summary = {
    'error_rows_count': error_rows_count,
    'earliest_error_ts': earliest_error_ts
}

# Keep intermediate variables in kernel: ad_pairs, adservice_kpi_table, df_ad_errors, adservice_error_summary
adservice_kpi_table, adservice_error_summary

```
Out[9]:
```
Summary for adservice in window 2022-03-21 01:30–01:59+08:

- Per-KPI anomaly highlights (top up to 20 sorted by relative breach then anomaly_count):
  - 20 KPI pairs returned. Top entries include:
    1. runtime.jvm_threads_started — anomaly_count=2, earliest=01:55, max=0.583333, p95=0.229167, abs_breach=0.354167, rel_breach≈1.54545
    2. runtime.java_lang_Memory_ObjectPendingFinalizationCount — anomaly_count=3, earliest=01:42, max=0.416667, p95=0.333333, abs_breach=0.083333, rel_breach=0.25
    3. runtime.java_lang_OperatingSystem_SystemLoadAverage — anomaly_count=2, earliest=01:56, max≈1.1925, p95≈0.9599, abs_breach≈0.2326, rel_breach≈0.2423
    4. runtime.jvm_memory_pool_MB_used.Survivor_Space — anomaly_count=2, earliest=01:45, rel_breach≈0.1896
    5. app.grpc.mrt — anomaly_count=2, earliest=01:42, rel_breach≈0.1780
    6. runtime.java_lang_GarbageCollector_LastGcInfo_* entries — anomaly_count=2, show very large absolute breaches (e.g., abs_breach ≈ 29,878 and ≈5,128) but moderate relative breaches (~0.17 and ~0.023)
    7. Several other JVM/thread/memory metrics with smaller relative breaches down to ~0.011.
  - Most KPI anomaly counts are small (mostly 2 or 3), indicating isolated minutes exceed global P95 rather than persistent full-window anomalies.
  - Relative breaches are moderate for top KPIs (largest ~1.545); there are no zero-denominator special cases (">>1e6") among the top 20 for adservice.

- Error log evidence for adservice in the same window:
  - error_rows_count = 0 (no error log rows for adservice in 01:30–01:59+08)
  - earliest_error_ts = None

Interpretation / next steps:
- adservice shows multiple JVM- and GC-related KPIs briefly exceeding their global P95s; notable are large absolute GC-related spikes (check GC traces/heap dumps).
- No corresponding error-log entries for adservice in the window, so investigate metrics (GC, thread counts, memory pools) and traces for causes rather than relying on application error logs.

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window    global_p95    abs_breach  rel_breach
0                         runtime.jvm_threads_started              2 2022-03-21 01:55:00+08:00         5.833333e-01  2.291667e-01      0.354167    1.545450
1   runtime.java_lang_Memory_ObjectPendingFinaliza...              3 2022-03-21 01:42:00+08:00         4.166667e-01  3.333333e-01      0.083333    0.250000
2   runtime.java_lang_OperatingSystem_SystemLoadAv...              2 2022-03-21 01:56:00+08:00         1.192500e+00  9.599167e-01      0.232583    0.242295
3      runtime.jvm_memory_pool_MB_used.Survivor_Space              2 2022-03-21 01:45:00+08:00         2.501431e-01  2.102662e-01      0.039877    0.189649
4                                        app.grpc.mrt              2 2022-03-21 01:42:00+08:00         3.407299e-03  2.892448e-03      0.000515    0.177999
5   runtime.java_lang_GarbageCollector_LastGcInfo_...              2 2022-03-21 01:43:00+08:00         2.025420e+05  1.726632e+05  29878.766667    0.173046
6             runtime.jvm_threads_state.TIMED_WAITING              3 2022-03-21 01:55:00+08:00         6.000000e+00  5.416667e+00      0.583333    0.107692
7   runtime.java_lang_MemoryPool_CollectionUsage_u...              2 2022-03-21 01:43:00+08:00         2.365327e+05  2.185646e+05  17968.033333    0.082209
8   runtime.java_lang_MemoryPool_Usage_used.Surviv...              2 2022-03-21 01:43:00+08:00         2.365327e+05  2.185646e+05  17968.033333    0.082209
9          runtime.jvm_memory_pool_MB_used.Eden_Space              2 2022-03-21 01:32:00+08:00         5.105432e+00  4.868718e+00      0.236714    0.048619
10      runtime.java_lang_Threading_DaemonThreadCount              3 2022-03-21 01:55:00+08:00         1.600000e+01  1.541667e+01      0.583333    0.037838
11                         runtime.jvm_threads_daemon              3 2022-03-21 01:55:00+08:00         1.600000e+01  1.541667e+01      0.583333    0.037838
12            runtime.java_lang_Threading_ThreadCount              3 2022-03-21 01:55:00+08:00         1.800000e+01  1.741667e+01      0.583333    0.033493
13                        runtime.jvm_threads_current              3 2022-03-21 01:55:00+08:00         1.800000e+01  1.741667e+01      0.583333    0.033493
14   runtime.java_lang_OperatingSystem_ProcessCpuLoad              2 2022-03-21 01:43:00+08:00         1.883292e-04  1.836167e-04      0.000005    0.025665
15  runtime.java_lang_GarbageCollector_LastGcInfo_...              2 2022-03-21 01:43:00+08:00         2.280253e+05  2.228965e+05   5128.866667    0.023010
16  runtime.java_lang_GarbageCollector_LastGcInfo_...              2 2022-03-21 01:34:00+08:00         1.833333e+00  1.795833e+00      0.037500    0.020882
17                                     app.grpc.count              2 2022-03-21 01:58:00+08:00         1.520000e+02  1.492000e+02      2.800000    0.018767
18  runtime.java_lang_MemoryPool_Usage_used.Eden_S...              2 2022-03-21 01:37:00+08:00         5.068553e+06  5.006840e+06  61712.233333    0.012326
19                                        app.grpc.sr              2 2022-03-21 01:33:00+08:00         7.714286e+01  7.627438e+01      0.868474    0.011386, {'error_rows_count': 0, 'earliest_error_ts': None})```
```